In [2]:
pip install duckdb pandas pyarrow datasets huggingface_hub

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
    --------------------------------------- 0.3/13.2 MB ? eta -:--:--
   -- ------------------------------------- 0.8/13.2 MB 1.8 MB/s eta 0:00:07
   ---- ----------------------------------- 1.6/13.2 MB 2.5 MB/s eta 0:00:05
   ------- -------------------------------- 2.4/13.2 MB 2.9 MB/s eta 0:00:04
   ---------- ----------------------------- 3.4/13.2 MB 3.6 MB/s eta 0:00:03
   ---------- ----------------------------- 3.4/13.2 MB 3.6 MB/s eta 0:00:03
   ---------- ----------------------------- 3.4/13.2 MB 3.6 MB/s eta 0:00:03
   ----------- ---------------------------- 3.9/13.2 MB 2.4 MB/s eta 0:00:04
   -------------- ------------------------- 4.7/13.2 MB 2.6 MB/s eta 0:00:04
   --------------- -------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import duckdb
import pandas as pd
from datetime import datetime

print("Setup complete")

Setup complete


In [5]:
from dotenv import load_dotenv
import os

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [6]:
import duckdb
import os

con = duckdb.connect()

token = os.getenv("HF_TOKEN")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{token}'
)
""")

print("DuckDB Hugging Face connection ready")

DuckDB Hugging Face connection ready


In [10]:
dataset_path = "hf://datasets/FlyRank/internship-warehouse"

print(dataset_path)

hf://datasets/FlyRank/internship-warehouse


In [11]:
query = f"""
SELECT *
FROM read_parquet(
'{dataset_path}/fact_content_daily_performance/**/*.parquet'
)
LIMIT 5
"""

df_test = con.sql(query)

df_test

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

In [13]:
print(HF_TOKEN[:5])

hf_Vi


In [12]:
df_test.columns

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row represents the daily performance record of one anonymized client and one anonymized content item on one report date.

The analysis window used is March 2026 (`month='2026-03'`).

March 2026 is selected as a mid-panel development month and avoids using the final month as a development period.

In [17]:
columns = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
'{dataset_path}/fact_content_daily_performance/**/*.parquet'
)
""")

columns

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [18]:
query_grain = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(
        DISTINCT 
        client_hash_id || '-' || content_hash_id || '-' || CAST(report_date AS VARCHAR)
    ) AS unique_grain_rows
FROM read_parquet(
'{dataset_path}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
"""

grain_result = con.sql(query_grain)

grain_result

┌────────────┬───────────────────┐
│ total_rows │ unique_grain_rows │
│   int64    │       int64       │
├────────────┼───────────────────┤
│    9841378 │           9841378 │
└────────────┴───────────────────┘

### Grain verification result

The query returned the same value for total rows and unique grain rows (9,841,378).

This confirms that each row represents one anonymized client, one anonymized content item, and one report date.

The main table used for this analysis is:

`fact_content_daily_performance`

This table contains daily performance observations for anonymized clients and content items.

## 2. Fields: feature / label / context / excluded

### Features

The following fields will be used as historical signals for content refresh opportunity scoring:

- `gsc_impressions`  
  Available at decision time because historical Search Console impressions are already observed.

- `gsc_clicks`  
  Available at decision time because historical clicks are recorded before making a refresh decision.

- `gsc_sum_position`  
  Available at decision time because historical search ranking information is available.

- `sessions_ai`  
  Available at decision time because historical AI referral sessions are observed.

- `scroll_events`  
  Available at decision time because historical user engagement events are recorded.

---

### Label

The target outcome is a future content refresh opportunity signal.

The label represents the desired future decision: whether content may need improvement based on later performance.

---

### Context

The following fields provide identification and filtering context:

- `client_hash_id`  
  An anonymized client identifier.

- `content_hash_id`  
  An anonymized content item identifier.

- `report_date`  
  The date of the observation.

- `month`  
  The partition month used for selecting the analysis window.

---

### Excluded

The following fields are excluded:

- Future performance metrics  
  Excluded because they would not be available at prediction time.

- Label-derived fields  
  Excluded because they would create data leakage and produce unrealistic model performance.

- Client or content identifying information  
  Excluded because the dataset is anonymized and outputs should not expose private information.

In [19]:
query_window = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
'{dataset_path}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
"""

window_result = con.sql(query_window)

window_result

┌───────────┬────────────┬────────────┐
│ row_count │ start_date │  end_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

### Row count and date window verification

The query shows 9,841,378 rows for the March 2026 slice.

The observed date range is from 2026-03-01 to 2026-03-31, confirming that the selected month contains the complete March 2026 reporting window.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [20]:
query_availability = f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet(
'{dataset_path}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
AND gsc_data_available IS TRUE
"""

availability_result = con.sql(query_availability)

availability_result

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│        3611061 │
└────────────────┘

### Availability verification

The query filters rows where `gsc_data_available IS TRUE`.

For March 2026, 3,611,061 rows have Search Console data available.

This confirms the availability of GSC data in the selected development slice.

## 4. Data limits

This dataset has several limitations:

- The history is unbalanced because different clients have different available data periods.

- Some early rows may contain Search Console data without complete Analytics information.

- Daily performance windows may overlap with rolling metrics, so observations are not always fully independent.

- The data can support decision-making and ranking opportunities, but it cannot prove that refreshing content will directly cause better performance.

## 5. Five feature frame
For the content refresh opportunity scoring lane, I selected five historical features that are available before making a refresh decision.

In [21]:
feature_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,

    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    sessions_ai,
    scroll_events

FROM read_parquet(
'{dataset_path}/fact_content_daily_performance/**/*.parquet'
)

WHERE month='2026-03'
LIMIT 100000
"""

feature_df = con.sql(feature_query).df()

feature_df.head()

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_sum_position,sessions_ai,scroll_events
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,67,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0,<NA>,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,616,<NA>,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,28,<NA>,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,25,<NA>,<NA>


### Feature availability

| Feature | Available when? |
|---|---|
| `gsc_impressions` | Available at decision time because historical Search Console impressions are already observed. |
| `gsc_clicks` | Available at decision time because historical clicks are recorded before deciding whether content needs refresh. |
| `gsc_sum_position` | Available at decision time because previous search ranking information is available from historical data. |
| `sessions_ai` | Available at decision time because historical AI referral sessions have already occurred. |
| `scroll_events` | Available at decision time because historical engagement events are recorded before the decision moment. |

### Missing value verification

The query shows that some engagement-related fields are not available for all observations.

In the March 2026 slice:
- `sessions_ai` is missing in 3,018,741 rows.
- `scroll_events` is missing in 3,018,741 rows.

These missing values reflect incomplete availability of some analytics signals and should be considered when building features.

In [22]:
missing_check = f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE sessions_ai IS NULL
    ) AS missing_sessions_ai,

    COUNT(*) FILTER (
        WHERE scroll_events IS NULL
    ) AS missing_scroll_events

FROM read_parquet(
'{dataset_path}/fact_content_daily_performance/**/*.parquet'
)

WHERE month='2026-03'
"""

con.sql(missing_check)

┌────────────┬─────────────────────┬───────────────────────┐
│ total_rows │ missing_sessions_ai │ missing_scroll_events │
│   int64    │        int64        │         int64         │
├────────────┼─────────────────────┼───────────────────────┤
│    9841378 │             3018741 │               3018741 │
└────────────┴─────────────────────┴───────────────────────┘

In [ ]:
# Feature dataframe copy karo
leak_df = feature_df.copy()

leak_df.head()

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_sum_position,sessions_ai,scroll_events
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,67,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0,<NA>,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,616,<NA>,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,28,<NA>,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,25,<NA>,<NA>


In [ ]:
# Fake label create karo
leak_df["refresh_label"] = (
    leak_df["gsc_clicks"] < 1
).astype(int)

leak_df.head()

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_sum_position,sessions_ai,scroll_events,refresh_label
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,67,<NA>,<NA>,1
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0,<NA>,<NA>,1
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,616,<NA>,<NA>,0
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,28,<NA>,<NA>,1
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,25,<NA>,<NA>,1


In [ ]:
# Intentional leakage feature add karo
leak_df["leaked_feature"] = leak_df["refresh_label"]

In [ ]:
# Show perfect relationship
leak_df[["refresh_label","leaked_feature"]].corr()

,refresh_label,leaked_feature
refresh_label,1.0,1.0
leaked_feature,1.0,1.0


In [28]:
from sklearn.metrics import accuracy_score

score = accuracy_score(
    leak_df["refresh_label"],
    leak_df["leaked_feature"]
)

print("Leakage accuracy:", score)

Leakage accuracy: 1.0


### Leakage experiment

A label-derived feature was intentionally added to demonstrate data leakage.

The leakage feature produced a perfect accuracy score of 1.0 because it directly contained the target information.

This feature was removed because it would not exist at prediction time.

Real model development should only use information available before the decision moment.

In [ ]:
# Remove leakage
clean_df = leak_df.drop(columns=["leaked_feature"])

clean_df.head()

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_sum_position,sessions_ai,scroll_events,refresh_label
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,67,<NA>,<NA>,1
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0,<NA>,<NA>,1
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,616,<NA>,<NA>,0
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,28,<NA>,<NA>,1
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,25,<NA>,<NA>,1


## Self-check

Before submission:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`